In [1]:
import pandas as pd
import numpy as np

# Load the data generated from Notebook 1
df = pd.read_csv("../data/processed/clean_data.csv")

# Display the percentage of missing values per column to map your targets
missing_percentages = df.isnull().mean() * 100
#print("Missing Value Percentages:\n", missing_percentages[missing_percentages > 0])

In [2]:
# Fill categorical values by make_model mode:
cat_cols = df.select_dtypes(include="object").columns

for col in cat_cols:
    for group in df["make_model"].dropna().unique():
        cond = df["make_model"] == group
        mode_val = df.loc[cond, col].mode()
        if not mode_val.empty:
            df.loc[cond, col] = df.loc[cond, col].fillna(mode_val[0])

In [3]:
#Fill remaining categorical values by make_model
for col in cat_cols:
    df[col] = df.groupby("make_model")[col].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan)
    )

In [4]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns

for col in num_cols:
    df[col] = df.groupby(["make_model", "age"])[col].transform(
        lambda x: x.fillna(x.median())
    )

In [5]:
# Fill categorical columns by make_model and age
cat_cols = df.select_dtypes(include="object").columns

for col in cat_cols:
    df[col] = df.groupby(["make_model", "age"])[col].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan)
    )

In [6]:
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [7]:
df.to_csv("../data/processed/missing_handled_data.csv", index=False)